## EXERCISE:
We have provided a spreadsheet of Boston housing prices (data/boston_housing.csv). The 14 columns are as follows:
1. CRIM: per capita crime rate by town
2. ZN: proportion of residential land zoned for lots over 25,000 sq.ft.
3. INDUS: proportion of non-retail business acres per town
4. CHAS: Charles River dummy variable (= 1 if tract bounds river; 0 otherwise)
5. NOX: nitric oxides concentration (parts per 10 million)
6. RM: average number of rooms per dwelling
7. AGE: proportion of owner-occupied units built prior to 1940
8. DIS: weighted distances to ﬁve Boston employment centers
9. RAD: index of accessibility to radial highways
10. TAX: full-value property-tax rate per \$10,000
11. PTRATIO: pupil-teacher ratio by town
12. B: 1000(Bk−0.63)2 where Bk is the proportion of blacks by town
13. LSTAT: % lower status of the population
14. MEDV: Median value of owner-occupied homes in $1000s We can see that the input attributes have a mixture of units

There are 450 observations.
<div class="exercise"><b>Exercise</b></div>

Using the above file, try your best to predict **housing prices. (the 14th column)** We have provided a test set `data/boston_housing_test.csv` but refrain from looking at the file or evaluating on it until you have finalized and trained a model.
1. Load in the data. It is tab-delimited. Quickly look at a summary of the data to familiarize yourself with it and ensure nothing is too egregious.
2. Use a previously-discussed function to automatically partition the data into a training and validation (aka development) set. It is up to you to choose how large these two portions should be.
3. Train a basic model on just a subset of the features. What is the performance on the validation set?
4. Train a basic model on all of the features. What is the performance on the validation set?
5. Toy with the model until you feel your results are reasonably good.
6. Perform cross-validation with said model, and measure the average performance. Are the results what you expected? Were the average results better or worse than that from your original 1 validation set?
7. Experiment with other models, and for each, perform 10-fold cross-validation. Which model yields the best average performance? Select this as your final model.
8. Use this model to evaulate your performance on the testing set. What is your performance (MSE)? Is this what you expected?

**1. Load in the data. It is tab-delimited. Quickly look at a summary of the data to familiarize yourself with it and ensure nothing is too egregious.**

In [ ]:
# My Code

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

# Loading the dataset (tab-delimited)
boston_train = pd.read_csv("/content/drive/MyDrive/boston_housing_train.csv", delim_whitespace=True, header=None)
boston_test = pd.read_csv("/content/drive/MyDrive/boston_housing_test.csv", delim_whitespace=True, header=None)

# Assign correct column names
columns = [
    "CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
    "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV"
]
boston_train.columns = columns
boston_test.columns = columns

# Display basic information about the dataset
print(boston_train.shape)
print(boston_train.head())
print(boston_train.columns)

(450, 14)
      CRIM    ZN  INDUS  CHAS    NOX     RM   AGE     DIS  RAD    TAX  \
0  0.00632  18.0   2.31     0  0.538  6.575  65.2  4.0900    1  296.0   
1  0.02731   0.0   7.07     0  0.469  6.421  78.9  4.9671    2  242.0   
2  0.02729   0.0   7.07     0  0.469  7.185  61.1  4.9671    2  242.0   
3  0.03237   0.0   2.18     0  0.458  6.998  45.8  6.0622    3  222.0   
4  0.06905   0.0   2.18     0  0.458  7.147  54.2  6.0622    3  222.0   

   PTRATIO       B  LSTAT  MEDV  
0     15.3  396.90   4.98  24.0  
1     17.8  396.90   9.14  21.6  
2     17.8  392.83   4.03  34.7  
3     18.7  394.63   2.94  33.4  
4     18.7  396.90   5.33  36.2  
Index(['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX',
       'PTRATIO', 'B', 'LSTAT', 'MEDV'],
      dtype='object')


In [ ]:
# Check for missing values
print("\nMissing values in training data:")
print(boston_train.isnull().sum())


Missing values in training data:
0.00632  18.00   2.310  0  0.5380  6.5750  65.20  4.0900   1  296.0  15.30 396.90   4.98  24.00    0
dtype: int64


**2. Use a previously-discussed function to automatically partition the data into a training and validation (aka development) set. It is up to you to choose how large these two portions should be.**

In [ ]:
# My Code

from sklearn.model_selection import train_test_split

X = boston_train.drop("MEDV", axis=1)
y = boston_train["MEDV"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_val.shape)


Training set shape: (360, 13)
Validation set shape: (90, 13)


**3. Train a basic model on just a subset of the features. What is the performance on the validation set?**

In [ ]:
# My Code

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Select a few important features
subset_features = ["RM", "LSTAT", "PTRATIO"]

# Subset training and validation data
X_train_subset = X_train[subset_features]
X_val_subset = X_val[subset_features]

# Train a simple linear regression model
linreg_subset = LinearRegression()
linreg_subset.fit(X_train_subset, y_train)

# Predict on validation data
y_pred_subset = linreg_subset.predict(X_val_subset)

# Evaluate model performance
mse_subset = mean_squared_error(y_val, y_pred_subset)
r2_subset = r2_score(y_val, y_pred_subset)

print("Subset Model MSE:", round(mse_subset, 2))
print("Subset Model R²:", round(r2_subset, 3))



Subset Model MSE: 25.75
Subset Model R²: 0.723


**4. Train a basic model on all of the features. What is the performance on the validation set?**

In [ ]:
# My Code

# Train on all 13 features
linreg_all = LinearRegression()
linreg_all.fit(X_train, y_train)

# Predict on validation data
y_pred_all = linreg_all.predict(X_val)

# Evaluate performance
mse_all = mean_squared_error(y_val, y_pred_all)
r2_all = r2_score(y_val, y_pred_all)

print("All Features MSE:", round(mse_all, 2))
print("All Features R²:", round(r2_all, 3))

All Features MSE: 21.5
All Features R²: 0.769


**5. Toy with the model until you feel your results are reasonably good.**

In [ ]:
# My Code

from sklearn.model_selection import cross_val_score
import numpy as np

# Reinitialize model
linreg_cv = LinearRegression()

# Perform 10-fold cross-validation
cv_mse = -cross_val_score(linreg_cv, X, y, cv=10, scoring='neg_mean_squared_error')
cv_r2 = cross_val_score(linreg_cv, X, y, cv=10, scoring='r2')

print("Average CV MSE:", round(np.mean(cv_mse), 2))
print("Average CV R²:", round(np.mean(cv_r2), 3))

Average CV MSE: 38.32
Average CV R²: 0.348


In [ ]:
from sklearn.linear_model import Ridge, Lasso

ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.1)

ridge_scores = cross_val_score(ridge, X, y, cv=10, scoring='r2')
lasso_scores = cross_val_score(lasso, X, y, cv=10, scoring='r2')

print("Average Ridge R²:", round(np.mean(ridge_scores), 3))
print("Average Lasso R²:", round(np.mean(lasso_scores), 3))

Average Ridge R²: 0.387
Average Lasso R²: 0.397


**6. Perform cross-validation with said model, and measure the average performance. Are the results what you expected? Were the average results better or worse than that from your original 1 validation set?**

Average Cross-Validation MSE: 38.32

Average Cross-Validation R²: 0.348

Previous Single Validation R²: 0.769, MSE: 21.5

The average cross-validation results were worse than the single validation results, which aligns with expectations. The model’s true predictive power appears to be closer to R² ≈ 0.35–0.40, rather than the initial 0.77 observed on one validation split.

**7. Experiment with other models, and for each, perform 10-fold cross-validation. Which model yields the best average performance? Select this as your final model.**

In [ ]:
# My Code

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

# Define models
models = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "Poly2_LinearRegression": Pipeline([
        ("poly", PolynomialFeatures(degree=2)),
        ("linear", LinearRegression())
    ])
}

# Perform 10-fold CV
cv_results = {}
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for name, model in models.items():
    mse_scores = -cross_val_score(model, X, y, cv=kf, scoring="neg_mean_squared_error")
    r2_scores = cross_val_score(model, X, y, cv=kf, scoring="r2")
    cv_results[name] = {
        "Average MSE": np.mean(mse_scores),
        "Average R²": np.mean(r2_scores)
    }

# Display results
for model_name, metrics in cv_results.items():
    print(f"{model_name}: Average MSE = {metrics['Average MSE']:.2f}, Average R² = {metrics['Average R²']:.3f}")


LinearRegression: Average MSE = 24.94, Average R² = 0.703
Ridge: Average MSE = 25.09, Average R² = 0.703
Lasso: Average MSE = 25.93, Average R² = 0.695
RandomForest: Average MSE = 11.67, Average R² = 0.868
Poly2_LinearRegression: Average MSE = 14.79, Average R² = 0.819


Linear, Ridge, Lasso: R² around 0.70–0.70, MSE ~25 — these are reasonable for linear models but not outstanding.

Polynomial (degree 2): R² = 0.819, MSE = 14.79 — better than plain linear, shows some non-linear relationships help.

Random Forest: R² = 0.868, MSE = 11.67 — the best performance by far, capturing non-linearities and interactions.

**8. Use this model to evaulate your performance on the testing set. What is your performance (MSE)? Is this what you expected?**

In [ ]:
# My Code

# Separate features and target for the test set
X_test = boston_test.drop("MEDV", axis=1)
y_test = boston_test["MEDV"]

# Train Random Forest on the full training set (all features)
rf_final = RandomForestRegressor(n_estimators=100, random_state=42)
rf_final.fit(X, y)  # X and y are the full training data from boston_train

# Make predictions on the test set
y_test_pred = rf_final.predict(X_test)

# Evaluate performance
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("Test Set MSE:", test_mse)
print("Test Set R²:", test_r2)

Test Set MSE: 12.54685210714285
Test Set R²: 0.31326344174902376


No it's not. The MSE is close to what I saw in CV, but the R² is surprisingly low (0.31). That’s much lower than the CV R² (~0.87).